In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime, gc
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Cấu hình đường dẫn chuẩn của Leader
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Khởi tạo Spark
spark = SparkSession.builder \
    .appName("HM_Trending_W8_Final") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "10g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("✅ Spark Ready! Hệ thống đã sẵn sàng tạo 'Phao cứu sinh' cho Tuần 8.")

Mounted at /content/drive
✅ Spark Ready! Hệ thống đã sẵn sàng tạo 'Phao cứu sinh' cho Tuần 8.


In [ ]:
# 1. Đọc dữ liệu
transactions = spark.read.parquet(INPUT_FILE)

# 2. Tính toán mốc thời gian (Logic dịch chuyển 1 tuần)
max_date = transactions.select(F.max("t_dat")).collect()[0][0]
test_start_date = max_date - datetime.timedelta(days=7)         # Đầu Tuần 8 (Mốc dự báo)
val_start_date = test_start_date - datetime.timedelta(days=7)   # Đầu Tuần 7 (Dùng tính Trending cho W8)

print(f"📅 Ngày cuối cùng dữ liệu: {max_date}")
print(f"📊 Tính Trending dựa trên Tuần 7: {val_start_date} -> {test_start_date}")
print(f"🎯 Mục tiêu dự báo cho Tuần 8: {test_start_date} -> {max_date}")

📅 Ngày cuối cùng dữ liệu: 2020-09-21 17:00:00
📊 Tính Trending dựa trên Tuần 7: 2020-09-07 17:00:00 -> 2020-09-14 17:00:00
🎯 Mục tiêu dự báo cho Tuần 8: 2020-09-14 17:00:00 -> 2020-09-21 17:00:00


In [ ]:
# 1. Lọc giao dịch trong Tuần 7 (Tuần ngay trước Tuần 8)
weekly_sales = transactions.filter(
    (F.col("t_dat") >= F.lit(val_start_date)) &
    (F.col("t_dat") < F.lit(test_start_date))
)

# 2. Đếm và lấy Top 12
top_12_trending = weekly_sales.groupBy("article_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(12) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))

# Chuyển thành Python List
trending_list = [row['article_id'] for row in top_12_trending.collect()]

print(f"🔥 Danh sách 12 món Trending 'vàng' của Tuần 7 để cứu cánh Tuần 8:")
for i, item in enumerate(trending_list):
    print(f"   Rank {i+1}: {item}")

🔥 Danh sách 12 món Trending 'vàng' của Tuần 7 để cứu cánh Tuần 8:
   Rank 1: 0909370001
   Rank 2: 0865799006
   Rank 3: 0918522001
   Rank 4: 0448509014
   Rank 5: 0751471001
   Rank 6: 0924243001
   Rank 7: 0918292001
   Rank 8: 0762846027
   Rank 9: 0863646001
   Rank 10: 0809238001
   Rank 11: 0715624001
   Rank 12: 0673677002


In [ ]:
# 1. Lấy danh sách khách hàng duy nhất
customers_df = transactions.select("customer_id").distinct()

# 2. Gán mảng 12 món Trending cho mỗi khách hàng
trending_candidates_df = customers_df.withColumn(
    "trending_candidates",
    F.array([F.lit(x) for x in trending_list])
)

# 3. LƯU FILE VỚI HẬU TỐ _W8
trending_candidates_df.write.mode("overwrite").parquet(OUTPUT_DIR + "trending_candidates_W8.parquet")

print(f"✅ Đã tạo xong file ứng viên Trending W8 cho {trending_candidates_df.count():,} khách hàng.")
print(f"📍 File lưu tại: {OUTPUT_DIR}trending_candidates_W8.parquet")

✅ Đã tạo xong file ứng viên Trending W8 cho 1,362,281 khách hàng.
📍 File lưu tại: /content/drive/MyDrive/HM-DATA/outputs/candidates/trending_candidates_W8.parquet


In [ ]:
# 1. Ground Truth: Thực tế khách mua ở Tuần 8
ground_truth_w8 = transactions.filter(
    (F.col("t_dat") >= F.lit(test_start_date))
).select("customer_id", F.lpad(F.col("article_id").cast("string"), 10, "0").alias("article_id"))

actual_counts = ground_truth_w8.groupBy("customer_id").count().withColumnRenamed("count", "actual_cnt")

# 2. Explode để tính Hits
candidates_exploded = trending_candidates_df.select(
    "customer_id",
    F.explode("trending_candidates").alias("article_id")
)

# 3. Join tìm món trùng khớp
hits = ground_truth_w8.join(candidates_exploded, ["customer_id", "article_id"], "inner") \
    .groupBy("customer_id").count().withColumnRenamed("count", "hit_cnt")

# 4. Tính Recall trung bình cho Tuần 8
recall_stats = actual_counts.join(hits, "customer_id", "left").fillna(0)
final_recall_trending_w8 = recall_stats.select(F.avg(F.col("hit_cnt") / F.col("actual_cnt"))).collect()[0][0]

print("-" * 55)
print(f"📊 KẾT QUẢ RECALL NHÁNH TRENDING TRÊN TUẦN 8")
print(f"Average Recall@12: {final_recall_trending_w8:.6f}")
print("-" * 55)

-------------------------------------------------------
📊 KẾT QUẢ RECALL NHÁNH TRENDING TRÊN TUẦN 8
Average Recall@12: 0.025973
-------------------------------------------------------
